In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import time
from multiprocessing import Pool
from tqdm.auto import tqdm
import re
from copy import deepcopy

import numpy as np
from scipy import integrate
from matplotlib import pyplot as plt

import noctiluca as nl
import bayesmsd

/home/sgh/gitlibs/chromatin_dynamics/.venv_SD_py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filename = '/data/sgh/science/2024_minflux/20260106_chromatin_dynamics_all_data.h5'
data       = nl.io.load.hdf5(filename)['data']

In [3]:
n_subsample = 4 # cut off the "kink" at the beginning of MINFLUX data
def subsample(traj):
    out = nl.Trajectory(traj[::n_subsample])
    out.meta['Δt'] = n_subsample*traj.meta['Δt']
    return out

In [4]:
data.makeSelection('minflux')
data.apply(subsample, inplace=True)

# Fits

In [5]:
def chop(traj, dt=None, L=200, Fmin=2):
    if dt is None:
        dt = traj.meta['Δt']
    
    def chop_traj(traj, dt=dt):
        if 'Δt' in traj.meta:
            dt = traj.meta['Δt']
            
        chops = []
        i0 = 0
        while i0 < len(traj):
            i1 = i0+L
            chop = traj.data[:, i0:min(i1, len(traj)), :]
            try:
                t_start = np.nonzero(~np.any(np.isnan(chop), axis=(0, 2)))[0][0]
            except IndexError: # no valid entries in this chop
                new_traj = nl.Trajectory(chop[:, [0]])
            else:
                new_traj = nl.Trajectory(chop[:, t_start:])
                
            new_traj.meta['Δt'] = dt
            chops.append(new_traj)

            i0 = i1
            
        return chops
    
    chops = chop_traj(traj)
    out = nl.TaggedSet(chops, hasTags=False)
    while len(chops) > 1:
        cg_traj = nl.Trajectory(np.stack([traj.data[:, 0] for traj in chops], axis=1))
        cg_traj.meta['Δt'] = L*chops[0].meta['Δt']
        chops = chop_traj(cg_traj)
        for traj in chops:
            out.add(traj)
    
    # Clean out useless trajectories
    out.makeSelection(lambda traj, _: traj.F < Fmin)
    out.deleteSelection()
    return out

In [6]:
ct = 'MEF'
bar = tqdm()

fits = {}
for treatment in ['ctrl']:

    cond = ['H2B', ct, treatment]
    fits[treatment] = {
        'single' : {},
        'joints' : {},
    }

    # Minflux
    data.makeSelection(['minflux', *cond], logic=all)
    dt = data[0].meta['Δt']

    fitdata = nl.TaggedSet()
    for traj in data:
        fitdata |= chop(traj.rescale(1e6, keepmeta=['Δt']))

    with nl.Parallelize():
        _ = nl.analysis.MSD(fitdata, chunksize=10, show_progress=True)

    fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=dt/n_subsample, parametrization='(log(αΓ), α)')
    fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
    fit.likelihood_chunksize = 200

    fits[treatment]['single'][f'minflux'] = fit

    bar.update()

    # Conventional
    for dt_tag in ['100ms', '2s']:
        data.makeSelection(['SPT', dt_tag, *cond], logic=all)
        dt = data[0].meta['Δt']
        tau_e = 0.08671 # same exposure for both conditions

        fitdata = data.apply(lambda traj : traj.relative(keepmeta=['MSD', 'Δt']), inplace=False)

        fit = bayesmsd.lib.NPFit(fitdata, motion_blur_f=tau_e, parametrization='(log(αΓ), α)')
        fit.parameters['log(σ²) (dim 1)'].fix_to = 'log(σ²) (dim 0)'
        fit.likelihood_chunksize = 100

        fits[treatment]['single'][f'SPT-{dt_tag}'] = fit

        bar.update()

    # Assemble list of fit(group)s to run
    groups = {
        'minflux'       : ['minflux'],
        'SPT 100ms'     : ['SPT-100ms'],
        'SPT 2s'        : ['SPT-2s'],
        'SPT'           : ['SPT-100ms', 'SPT-2s'],
        'minflux + SPT' : ['minflux', 'SPT-100ms', 'SPT-2s'],
    }

    for groupname in groups:
        fits_dict = fits[treatment]['single']

        fit = bayesmsd.FitGroup({name : fits_dict[name] for name in groups[groupname]})
        fit.parameters['α']       = deepcopy(fits_dict['minflux'].parameters[      'α (dim 0)'])
        fit.parameters['log(αΓ)'] = deepcopy(fits_dict['minflux'].parameters['log(αΓ) (dim 0)'])

        # hacky...
        def patch_initial_params(self=fit):
            params = type(self).initial_params(self)
            a    = [val for key, val in params.items() if      'α' in key][0]
            logG = [val for key, val in params.items() if 'log(αΓ)' in key][0]
            params['α'] = a
            params['log(αΓ)'] = logG
            return params
        fit.initial_params = patch_initial_params

        for fitname in fit.fits_dict:
            fit.parameters[fitname+f' α (dim 0)'].fix_to = 'α'
            if fitname == 'minflux':
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = 'log(αΓ)'
            else: # not minflux, so correct for 2-loc
                def twoGref(params): return params['log(αΓ)']+np.log(2)
                fit.parameters[fitname+f' log(αΓ) (dim 0)'].fix_to = twoGref

        fits[treatment]['joints'][groupname] = fit

        bar.update()

bar.close()

0it [00:00, ?it/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████| 4921/4921 [00:01<00:00, 2664.55it/s]
8it [00:04,  1.65it/s]


In [7]:
fitres = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    fitres[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)

        with nl.Parallelize():
            fitres[treatment][name] = fits[treatment]['joints'][name].run(show_progress=True)

        for key in fitres[treatment][name]['params']:
            print(key, fitres[treatment][name]['params'][key])
        print()


||   MEF ctrl  ||

minflux


fit iterations: 78it [00:51,  1.51it/s]


minflux log(σ²) (dim 0) -8.57741319358731
α 0.3112186063990455
log(αΓ) -6.128192099779255
minflux α (dim 0) 0.3112186063990455
minflux log(αΓ) (dim 0) -6.128192099779255

SPT 100ms


fit iterations: 58it [00:22,  2.52it/s]


SPT-100ms log(σ²) (dim 0) -7.312204047514058
α 0.3304676474053503
log(αΓ) -6.465116926229573
SPT-100ms α (dim 0) 0.3304676474053503
SPT-100ms log(αΓ) (dim 0) -5.771969745669628

SPT 2s


...

SPT-2s log(σ²) (dim 0) -6.108735295906271
α 0.48608507509048726
log(αΓ) -6.6560896392252635
SPT-2s α (dim 0) 0.48608507509048726
SPT-2s log(αΓ) (dim 0) -5.962942458665318

SPT


fit iterations: 71it [00:46,  1.54it/s]


SPT-100ms log(σ²) (dim 0) -7.2389341387226525
SPT-2s log(σ²) (dim 0) -7.031286288791447
α 0.3635523377604423
log(αΓ) -6.425612979295025
SPT-100ms α (dim 0) 0.3635523377604423
SPT-2s α (dim 0) 0.3635523377604423
SPT-100ms log(αΓ) (dim 0) -5.732465798735079
SPT-2s log(αΓ) (dim 0) -5.732465798735079

minflux + SPT


fit iterations: 193it [03:37,  1.13s/it]

minflux log(σ²) (dim 0) -8.601558420213966
SPT-100ms log(σ²) (dim 0) -7.695942802007502
SPT-2s log(σ²) (dim 0) -8.307385404817303
α 0.27646054814765825
log(αΓ) -6.3879235959277025
minflux α (dim 0) 0.27646054814765825
minflux log(αΓ) (dim 0) -6.3879235959277025
SPT-100ms α (dim 0) 0.27646054814765825
SPT-2s α (dim 0) 0.27646054814765825
SPT-100ms log(αΓ) (dim 0) -5.694776415367757
SPT-2s log(αΓ) (dim 0) -5.694776415367757



In [8]:
nl.io.write.hdf5(fitres, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')

## Profiler
Estimate credible intervals for point estimates from profile likelihood. __Attention: computationally expensive__

This can also move the point estimate, if we find better parameters while exploring

In [9]:
fitres = nl.io.load.hdf5(f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')
mci = {}
for treatment in ['ctrl']:
    print()
    print(17*'=')
    print(f'|| {ct:>5s} {treatment:<5s} ||')
    print(17*'=')
    print()
    
    mci[treatment] = {}
    for name in fits[treatment]['joints']:
        print(name)
        print('='*20)
        
        profiler = bayesmsd.Profiler(fits[treatment]['joints'][name], max_restarts=50)
        profiler.point_estimate = fitres[treatment][name]

        with nl.Parallelize():
            mci[treatment][name] = profiler.find_MCI(show_progress=True)

        for key in mci[treatment][name]:
            m, (cil, cih) = mci[treatment][name][key]
            print(f"{key:>25s} = {m:>6.3f} [{cil:>6.3f}, {cih:>6.3f}]")
        print()


||   MEF ctrl  ||

minflux


profiler iterations: 103it [30:51, 17.97s/it]


  minflux log(σ²) (dim 0) = -8.577 [-8.609, -8.548]
                        α =  0.311 [ 0.304,  0.318]
                  log(αΓ) = -6.128 [-6.168, -6.089]

SPT 100ms


profiler iterations: 82it [12:03,  8.83s/it]


SPT-100ms log(σ²) (dim 0) = -7.312 [-7.339, -7.287]
                        α =  0.330 [ 0.320,  0.340]
                  log(αΓ) = -6.465 [-6.482, -6.448]

SPT 2s


...

   SPT-2s log(σ²) (dim 0) = -6.109 [-6.159, -6.053]
                        α =  0.486 [ 0.472,  0.501]
                  log(αΓ) = -6.656 [-6.672, -6.629]

SPT


profiler iterations: 196it [1:12:59, 22.34s/it]


SPT-100ms log(σ²) (dim 0) = -7.239 [-7.254, -7.223]
   SPT-2s log(σ²) (dim 0) = -7.031 [-7.085, -6.983]
                        α =  0.364 [ 0.359,  0.368]
                  log(αΓ) = -6.426 [-6.433, -6.420]

minflux + SPT


profiler iterations: 269it [4:21:13, 58.27s/it] 

  minflux log(σ²) (dim 0) = -8.602 [-8.618, -8.582]
SPT-100ms log(σ²) (dim 0) = -7.696 [-7.712, -7.680]
   SPT-2s log(σ²) (dim 0) = -8.307 [-8.497, -8.143]
                        α =  0.276 [ 0.275,  0.278]
                  log(αΓ) = -6.388 [-6.393, -6.383]



In [10]:
nl.io.write.hdf5(mci, f'/data/sgh/science/2024_minflux/fits/20260106_fitres_NPFit-aGparam_{ct}.h5')